# Manzil v2 — XLM-R vs mBERT benchmark (Google Colab)

This notebook fine-tunes **XLM-RoBERTa-base** and **mBERT** on the Manzil v2 pilot for:
1. Intent classification (NAVIGATE / ETA_QUERY / SEARCH / OTHER)
2. Token-level entity extraction (BIO tags)

It evaluates both models on the same pre-defined train/validation/test split and reports intent accuracy/macro-F1, NER token metrics, exact entity-span precision/recall/F1, per-language results, confusion matrices, and runtime.

**Research integrity:** the current pilot contains synthetic candidate examples. Results from this notebook are preliminary and must not be described as real-world or human-validated performance. For final research claims, rerun evaluation on the adjudicated human-reviewed test set after completing the annotation workbook.

**Compute:** use Colab Runtime → Change runtime type → GPU. This may take a while; train one model first if GPU memory/time is limited.

In [ ]:
!pip -q install transformers datasets accelerate seqeval scikit-learn pandas numpy matplotlib seaborn

## 1. Upload the Manzil project ZIP

Upload `Manzil_v2_Final_Package.zip` (or a ZIP containing the three JSONL datasets). The notebook extracts it and loads the token-level and sentence-level files.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose Manzil_v2_Final_Package.zip
import os, zipfile, glob, json, time, random, numpy as np, pandas as pd, torch
zip_path = next((os.path.abspath(k) for k in uploaded if k.lower().endswith('.zip')), None)
assert zip_path, "Please upload the Manzil project ZIP."
EXTRACT_DIR = "/content/manzil_data"
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(zip_path) as z: z.extractall(EXTRACT_DIR)
token_path = glob.glob(EXTRACT_DIR + "/**/manzil_v2_token_level.jsonl", recursive=True)[0]
full_path = glob.glob(EXTRACT_DIR + "/**/manzil_v2_full.jsonl", recursive=True)[0]
tok = [json.loads(x) for x in open(token_path, encoding="utf-8")]
full = [json.loads(x) for x in open(full_path, encoding="utf-8")]
print("Token rows:", len(tok), "Full rows:", len(full), "GPU:", torch.cuda.is_available())

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from seqeval.metrics import precision_score as span_precision, recall_score as span_recall, f1_score as span_f1, classification_report as span_report
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          AutoModelForSequenceClassification, get_linear_schedule_with_warmup)
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from collections import defaultdict
import matplotlib.pyplot as plt, seaborn as sns

SEED=429
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
intent_labels=sorted({r["intent"] for r in full})
intent2id={x:i for i,x in enumerate(intent_labels)}
id2intent={i:x for x,i in intent2id.items()}
bio_labels=sorted({tag for r in tok for tag in r["bio_tags"]})
bio2id={x:i for i,x in enumerate(bio_labels)}
id2bio={i:x for x,i in bio2id.items()}
print("Intent labels:", intent2id)
print("BIO labels:", bio2id)

def split_rows(rows):
    return {s:[r for r in rows if r.get("split")==s] for s in ["train","validation","test"]}
tok_s=split_rows(tok); full_s=split_rows(full)
for s in tok_s: print(s, len(tok_s[s]), len(full_s[s]))
assert all(tok_s[s] and full_s[s] for s in ["train","validation","test"]), "Missing split; inspect source split labels."


## 2. Dataset classes and training/evaluation helpers

The notebook uses word-level BIO labels and aligns them to subword tokens with `word_ids()`. Special tokens and continuation subwords are ignored in the loss. Exact-span F1 is calculated with `seqeval`.

In [ ]:
class IntentDS(Dataset):
    def __init__(self, rows, tokenizer, max_len=128):
        self.rows=rows; self.enc=tokenizer([r["query"] for r in rows], truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        x={k:v[i] for k,v in self.enc.items()}
        x["labels"]=torch.tensor(intent2id[self.rows[i]["intent"]],dtype=torch.long)
        return x

class NERDS(Dataset):
    def __init__(self, rows, tokenizer, max_len=128):
        self.items=[]
        for r in rows:
            words=r["token_sequence"]; tags=r["bio_tags"]
            enc=tokenizer(words, is_split_into_words=True, truncation=True, padding="max_length", max_length=max_len)
            word_ids=enc.word_ids()
            labs=[]; prev=None
            for wid in word_ids:
                if wid is None: labs.append(-100)
                elif wid != prev: labs.append(bio2id[tags[wid]])
                else: labs.append(-100)
                prev=wid
            self.items.append((enc, labs))
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        enc,labs=self.items[i]
        return {k:torch.tensor(v,dtype=torch.long) for k,v in enc.items()} | {"labels":torch.tensor(labs,dtype=torch.long)}

def predict_intent(model, loader):
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for b in loader:
            y=b.pop("labels").numpy()
            b={k:v.to(device) for k,v in b.items()}
            p=model(**b).logits.argmax(-1).cpu().numpy()
            ys.extend(y); ps.extend(p)
    return np.array(ys),np.array(ps)

def predict_ner(model, loader):
    model.eval(); true_seq=[]; pred_seq=[]
    with torch.no_grad():
        for b in loader:
            labels=b["labels"].numpy()
            inputs={k:v.to(device) for k,v in b.items() if k!="labels"}
            logits=model(**inputs).logits.argmax(-1).cpu().numpy()
            for gold,pred in zip(labels,logits):
                g=[]; p=[]
                for gi,pi in zip(gold,pred):
                    if gi!=-100:
                        g.append(id2bio[int(gi)]); p.append(id2bio[int(pi)])
                true_seq.append(g); pred_seq.append(p)
    return true_seq,pred_seq

def train_task(model, train_loader, val_loader, task, epochs=3, lr=2e-5):
    model.to(device); opt=AdamW(model.parameters(),lr=lr)
    total=epochs*len(train_loader)
    sched=get_linear_schedule_with_warmup(opt, num_warmup_steps=max(1,int(.1*total)), num_training_steps=total)
    best=-1; best_state=None
    for ep in range(epochs):
        model.train(); losses=[]
        for b in train_loader:
            b={k:v.to(device) for k,v in b.items()}
            opt.zero_grad(); out=model(**b); out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step(); sched.step(); losses.append(out.loss.item())
        if task=="intent":
            y,p=predict_intent(model,val_loader); score=f1_score(y,p,average="macro")
        else:
            y,p=predict_ner(model,val_loader); score=span_f1(y,p)
        print(f"{task} epoch {ep+1}/{epochs} loss={np.mean(losses):.4f} val_macro_or_span_f1={score:.4f}")
        if score>best: best=score; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best_state); return model

def intent_report(model, rows, tokenizer):
    loader=DataLoader(IntentDS(rows,tokenizer),batch_size=16)
    y,p=predict_intent(model,loader)
    return {"accuracy":float(accuracy_score(y,p)),"macro_f1":float(f1_score(y,p,average="macro")),
            "report":classification_report(y,p,labels=list(range(len(intent_labels))),target_names=intent_labels,output_dict=True,zero_division=0),
            "confusion_matrix":confusion_matrix(y,p,labels=list(range(len(intent_labels)))).tolist()}

def ner_report(model, rows, tokenizer):
    loader=DataLoader(NERDS(rows,tokenizer),batch_size=8)
    y,p=predict_ner(model,loader)
    return {"span_precision":float(span_precision(y,p)),"span_recall":float(span_recall(y,p)),
            "span_f1":float(span_f1(y,p)),"report":span_report(y,p,output_dict=True,zero_division=0)}


## 3. Fine-tune both model families

Default model IDs:
- XLM-R: `FacebookAI/xlm-roberta-base`
- mBERT: `google-bert/bert-base-multilingual-cased`

Set `MODELS_TO_RUN` to one model first if Colab resources are limited. The notebook fine-tunes separate sequence-classification and token-classification heads.

In [ ]:
MODEL_CONFIGS={
    "XLM-R":"FacebookAI/xlm-roberta-base",
    "mBERT":"google-bert/bert-base-multilingual-cased"
}
MODELS_TO_RUN=["XLM-R","mBERT"]  # change to ["XLM-R"] for a quicker first run
EPOCHS=3
all_results={}
os.makedirs("/content/manzil_transformer_models",exist_ok=True)

for model_key in MODELS_TO_RUN:
    model_name=MODEL_CONFIGS[model_key]
    print("\n==========",model_key,model_name,"==========")
    t0=time.time()
    tokenizer=AutoTokenizer.from_pretrained(model_name, use_fast=True)
    # Intent
    itr=DataLoader(IntentDS(full_s["train"],tokenizer),batch_size=16,shuffle=True)
    iva=DataLoader(IntentDS(full_s["validation"],tokenizer),batch_size=16)
    im=AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=len(intent_labels),id2label=id2intent,label2id=intent2id)
    im=train_task(im,itr,iva,"intent",epochs=EPOCHS)
    # NER
    ntr=DataLoader(NERDS(tok_s["train"],tokenizer),batch_size=8,shuffle=True)
    nva=DataLoader(NERDS(tok_s["validation"],tokenizer),batch_size=8)
    nm=AutoModelForTokenClassification.from_pretrained(model_name,num_labels=len(bio_labels),id2label=id2bio,label2id=bio2id)
    nm=train_task(nm,ntr,nva,"ner",epochs=EPOCHS)
    # Save model + tokenizer
    out=f"/content/manzil_transformer_models/{model_key}"
    os.makedirs(out,exist_ok=True)
    im.save_pretrained(out+"/intent"); nm.save_pretrained(out+"/ner"); tokenizer.save_pretrained(out+"/tokenizer")
    # Test evaluation (run only after model selection on validation)
    all_results[model_key]={
        "intent":intent_report(im,full_s["test"],tokenizer),
        "ner":ner_report(nm,tok_s["test"],tokenizer),
        "seconds":round(time.time()-t0,1)
    }
    print("Intent accuracy/macro-F1:",all_results[model_key]["intent"]["accuracy"],all_results[model_key]["intent"]["macro_f1"])
    print("NER exact-span P/R/F1:",all_results[model_key]["ner"]["span_precision"],all_results[model_key]["ner"]["span_recall"],all_results[model_key]["ner"]["span_f1"])


## 4. Per-language test evaluation

This helps reveal whether performance differs between Devanagari Hindi, Romanized Hindi and Hinglish.

In [ ]:
def per_language_reports(model_key):
    model_name=MODEL_CONFIGS[model_key]
    tokenizer=AutoTokenizer.from_pretrained(f"/content/manzil_transformer_models/{model_key}/tokenizer",use_fast=True)
    im=AutoModelForSequenceClassification.from_pretrained(f"/content/manzil_transformer_models/{model_key}/intent").to(device)
    nm=AutoModelForTokenClassification.from_pretrained(f"/content/manzil_transformer_models/{model_key}/ner").to(device)
    out={}
    for lang in sorted({r["language"] for r in full_s["test"]}):
        fr=[r for r in full_s["test"] if r["language"]==lang]
        tr=[r for r in tok_s["test"] if r["language"]==lang]
        out[lang]={"intent":intent_report(im,fr,tokenizer),"ner":ner_report(nm,tr,tokenizer)}
    return out

for k in MODELS_TO_RUN:
    all_results[k]["per_language"]=per_language_reports(k)


## 5. Save reports and model comparison table

The JSON contains the full classification reports and confusion matrices. The CSV is a concise comparison for the project report.

In [ ]:
summary=[]
for k,v in all_results.items():
    summary.append({"Model":k,"Intent accuracy":v["intent"]["accuracy"],"Intent macro-F1":v["intent"]["macro_f1"],
                    "NER exact-span precision":v["ner"]["span_precision"],"NER exact-span recall":v["ner"]["span_recall"],
                    "NER exact-span F1":v["ner"]["span_f1"],"Runtime seconds":v["seconds"]})
summary_df=pd.DataFrame(summary)
display(summary_df)
with open("/content/manzil_transformer_evaluation.json","w",encoding="utf-8") as f: json.dump(all_results,f,ensure_ascii=False,indent=2)
summary_df.to_csv("/content/manzil_transformer_comparison.csv",index=False)


## 6. Download results and trained models

The ZIP contains the transformer checkpoints, metrics, confusion matrices and comparison table. Keep the human-reviewed test results separate from the synthetic-pilot results.

In [ ]:
import shutil
shutil.make_archive("/content/Manzil_Transformer_Results","zip","/content/manzil_transformer_models")
with zipfile.ZipFile("/content/Manzil_Transformer_Results.zip","a",zipfile.ZIP_DEFLATED) as z:
    z.write("/content/manzil_transformer_evaluation.json","manzil_transformer_evaluation.json")
    z.write("/content/manzil_transformer_comparison.csv","manzil_transformer_comparison.csv")
files.download("/content/Manzil_Transformer_Results.zip")
files.download("/content/manzil_transformer_evaluation.json")
